# Lecture 5 · Notebook 1 — Tokenizers: BPE from scratch, and why your language may cost more tokens
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture05/lecture05_1_tokenizers_and_bpe.ipynb)

**Companion to [Chapter 6 of the video lecture](https://nlp-course-cuhksz.github.io/lecture05.html#p6-s83)** (slides 76–86). Before an LLM can read anything, text is cut into
tokens. Almost every modern LLM uses a *subword* tokenizer learned with some variant of **byte-pair encoding (BPE)**. In this notebook you will:

1. replay BPE as a compression algorithm on the string from slide 83,
2. implement BPE training on a toy corpus and split words the tokenizer has never seen,
3. train a real byte-level BPE tokenizer on Shakespeare and see how vocabulary size trades off against sequence length, and
4. compare the tokenizers of GPT-2, the original LLaMA and Qwen on English, Chinese and Arabic — and reproduce the
   "28 characters → 35 tokens" table of slide 84.

Runs on CPU in about three minutes. Downloads: three tokenizers (a few MB) and a 1 MB text file.

In [ ]:
!pip -q install transformers   # already installed on Colab; needed elsewhere

## 1 · BPE as compression (slide 83)
Repeatedly find the most frequent pair of adjacent symbols and replace it with a new symbol. Stop when no pair occurs twice.

In [ ]:
from collections import Counter

def pair_counts(seq):
    return Counter(zip(seq, seq[1:]))

def merge(seq, pair, new):
    out, i = [], 0
    while i < len(seq):
        if i + 1 < len(seq) and (seq[i], seq[i + 1]) == pair:
            out.append(new); i += 2          # replace the pair, left to right, without overlaps
        else:
            out.append(seq[i]); i += 1
    return out

seq, table = list("aaabdaaabac"), {}
print("start  :", "".join(seq))
for new in "ZYXWV":
    counts = pair_counts(seq)
    best = max(counts.values())
    if best < 2:
        break
    # ties: slide 83 (and Wikipedia) merge "ab" before "Za", so we prefer pairs made of fewer new symbols
    pair = max((p for p, c in counts.items() if c == best), key=lambda p: -sum(ch.isupper() for ch in p))
    seq = merge(seq, pair, new)
    table[new] = "".join(pair)
    print(f"{''.join(pair)!r:6s} -> {new}: {''.join(seq)}")
print("replacement table:", table)

# decompress: apply the replacements in reverse order
s = "".join(seq)
for new, pair in reversed(table.items()):
    s = s.replace(new, pair)
print("decoded:", s)

## 2 · BPE training on words
For tokenization we run the same loop on a corpus. Each word starts as a sequence of characters plus an end-of-word marker `</w>`, and
every merge becomes a new vocabulary entry. The ordered **list of merges is the tokenizer**. A toy corpus with word frequencies
(the classic example from Sennrich et al., 2016):

In [ ]:
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}

def learn_bpe(word_freqs, num_merges):
    words = {tuple(w) + ("</w>",): f for w, f in word_freqs.items()}
    merges = []
    for step in range(num_merges):
        counts = Counter()
        for w, f in words.items():
            for p in zip(w, w[1:]):
                counts[p] += f
        if not counts:
            break
        pair, n = counts.most_common(1)[0]
        merges.append(pair)
        words = {tuple(merge(list(w), pair, pair[0] + pair[1])): f for w, f in words.items()}
        print(f"merge {step + 1:2d}: {pair[0]!r} + {pair[1]!r}  (seen {n} times)   ->  {[' '.join(w) for w in words]}")
    return merges

merges = learn_bpe(corpus, 10)

Now apply the learned merges, in order, to words that were **not** in the corpus. No word is ever unknown: in the worst case it falls back to characters.

In [ ]:
def bpe_encode(word, merges):
    seq = list(word) + ["</w>"]
    for pair in merges:
        seq = merge(seq, pair, pair[0] + pair[1])
    return seq

for w in ["lowest", "newer", "wider", "slow", "zebra"]:
    print(f"{w:8s} -> {bpe_encode(w, merges)}")

## 3 · A real byte-level BPE tokenizer, trained on Shakespeare
The Hugging Face `tokenizers` library trains the same algorithm on bytes (so every string can be encoded) in a second.
We train several vocabulary sizes and measure **how many tokens a word needs** on held-out text: a bigger vocabulary means shorter
sequences but a larger embedding table — the trade-off of slides 79–80.

In [ ]:
import requests
from tokenizers import ByteLevelBPETokenizer

text = requests.get("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", timeout=60).text
split = int(0.9 * len(text))
train_text, test_text = text[:split], text[split:]
print(f"{len(text):,} characters; first lines:\n{text[:200]}")

In [ ]:
import matplotlib.pyplot as plt

n_words = len(test_text.split())
results = {}
for vocab in [300, 500, 1000, 2000, 4000, 8000]:
    tk = ByteLevelBPETokenizer()
    tk.train_from_iterator([train_text], vocab_size=vocab, min_frequency=2, show_progress=False)
    n_tok = len(tk.encode(test_text).ids)
    results[vocab] = (tk, n_tok / n_words)
    print(f"vocab {vocab:5d}: {n_tok / n_words:.2f} tokens per word,  'tokenizing' -> {tk.encode(' tokenizing').tokens}")

plt.figure(figsize=(5, 3.2))
plt.plot(list(results), [v[1] for v in results.values()], "o-")
plt.xscale("log"); plt.xlabel("vocabulary size"); plt.ylabel("tokens per word (held-out)")
plt.title("Bigger vocabulary, shorter sequences"); plt.grid(alpha=.3); plt.show()

The `Ġ` is how byte-level BPE shows a leading space. Frequent words become single tokens; rare ones, like *tokenizing* (which never
appears in Shakespeare), are split into pieces. The first merges a tokenizer learns tell you what is frequent in its corpus:

In [ ]:
tk = results[2000][0]
vocab = tk.get_vocab()
learned = sorted(vocab.items(), key=lambda kv: kv[1])[256:286]   # ids 0-255 are the raw bytes
print([t for t, _ in learned])
print(tk.encode("Wherefore art thou Romeo? Unfriendly tokenization!").tokens)

## 4 · Real tokenizers on English, Chinese and Arabic (slides 84–85)
We load only the **tokenizers** (no model weights) of three LLMs:

| tokenizer | vocabulary |
|---|---|
| GPT-2 (2019) | byte-level BPE, 50K tokens, trained mostly on English web text |
| original LLaMA (2023) | SentencePiece BPE, 32K tokens, falls back to raw UTF-8 **bytes** for unknown characters — LLaMA 2 and AceGPT use the same vocabulary |
| Qwen2.5 (2024) | byte-level BPE, about 151K tokens, built for many languages |

First, the exact sentence from slide 84.

In [ ]:
from transformers import AutoTokenizer
toks = {
    "GPT-2": AutoTokenizer.from_pretrained("gpt2"),
    "LLaMA": AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer"),   # the original LLaMA 32K vocabulary
    "Qwen2.5": AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B"),
}

def show(tok, s):
    ids = tok(s, add_special_tokens=False)["input_ids"]
    # byte-level tokens are shown decoded; a lone '\ufffd' means the token is only part of a character's UTF-8 bytes
    pieces = tok.convert_ids_to_tokens(ids) if "LLaMA" in tok.name_or_path or "llama" in tok.name_or_path else [tok.decode([i]) for i in ids]
    return ids, pieces

s84 = "人工智能是计算机科学、心理学、哲学等学科融合的交叉学科。"
print(f"original sentence: {len(s84)} characters\n")
for name, tok in toks.items():
    ids, pieces = show(tok, s84)
    print(f"{name:8s} {len(ids):3d} tokens: {pieces}\n")

The LLaMA row reproduces slide 84: **35 tokens for 28 characters**, with `<0xE5> <0x93> <0xB2>` — the three UTF-8 bytes of 哲, a character
missing from the vocabulary. GPT-2 is even worse; Qwen, whose vocabulary was built with Chinese in mind, needs 15 tokens. Now a few
sentences in three languages (each says roughly the same thing):

In [ ]:
samples = {
    "English": "Artificial intelligence is an interdisciplinary field that combines computer science, psychology and philosophy.",
    "Chinese": "人工智能是计算机科学、心理学、哲学等学科融合的交叉学科。",
    "Arabic":  "الذكاء الاصطناعي مجال متعدد التخصصات يجمع بين علوم الحاسوب وعلم النفس والفلسفة.",
}
import numpy as np
table = {name: [len(tok(s, add_special_tokens=False)["input_ids"]) for s in samples.values()] for name, tok in toks.items()}
print(f"{'':10s}" + "".join(f"{lang:>10s}" for lang in samples))
print(f"{'chars':10s}" + "".join(f"{len(s):10d}" for s in samples.values()))
for name, counts in table.items():
    print(f"{name:10s}" + "".join(f"{c:10d}" for c in counts))

x = np.arange(len(samples)); w = 0.27
plt.figure(figsize=(6, 3.2))
for k, (name, counts) in enumerate(table.items()):
    plt.bar(x + (k - 1) * w, counts, w, label=name)
plt.xticks(x, list(samples)); plt.ylabel("tokens for the same sentence"); plt.legend(); plt.show()

More tokens for the same meaning means slower and more expensive training and inference, a shorter effective context window, and a
harder learning problem (the model must reassemble characters from bytes). This is why Chinese LLaMA extended the vocabulary, and why
the AceGPT tokenizer was criticized for Arabic (slide 85).

## 5 · A side effect: models do not see letters

In [ ]:
for word in ["strawberry", " strawberry", "Strawberry", "STRAWBERRY"]:
    ids, pieces = show(toks["Qwen2.5"], word)
    print(f"{word!r:14s} -> {pieces}")

The model receives a few token ids, not ten letters — one reason why counting the *r*'s in "strawberry" was famously hard for LLMs.
Note also that the same word gets different tokens depending on case and the leading space.

## Exercises
1. Add a sentence in another language or script you know (Japanese, Korean, Hindi, Cantonese, emoji, Python code). Which tokenizer is most
   efficient, and why?
2. In section 3, train the tokenizer on Chinese text instead (paste a few paragraphs). How large must the vocabulary be before common
   two-character words become single tokens?
3. Load `hfl/chinese-llama-2-7b`'s tokenizer (`AutoTokenizer.from_pretrained(...)`; Colab already has `sentencepiece`). How many tokens does it
   need for the slide-84 sentence, compared with the 16 of the Chinese LLaMA tokenizer on the slide?
4. Section 1 had a tie between `Za` and `ab`. Break the tie the other way. Does the final compressed string get shorter or longer?

**Next:** [Notebook 2 — scaling laws: fit your own power law]({colab(N2)})